In [103]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso, ElasticNet
from sklearn.svm import LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np
import pandas as pd

PVGIS_filename = "data/Timeseries_59.246_18.035_SA3_1kWp_crystSi_14_42deg_0deg_2022_2023.csv"

def process_pvgis (filepath):
    df = pd.read_csv(filepath, header=8, nrows=17520)
    df['time'] = df['time'].astype(str).str[:-2]
    df['time'] = pd.to_datetime(df['time'], format='%Y%m%d:%H')
    df['day_of_year'] = df['time'].dt.dayofyear
    df['hour_of_day'] = df['time'].dt.hour

    df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    X = df[['G(i)', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']]
    y = df[['P']]

    X_train, X_test = X.iloc[:8760], X.iloc[8760:]
    y_train, y_test = y.iloc[:8760], y.iloc[8760:]
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = process_pvgis(PVGIS_filename)


In [104]:
X_train[:50]

,G(i),hour_sin,hour_cos,day_sin,day_cos
0,0.00,0.000000e+00,1.000000e+00,0.017213,0.999852
1,0.00,2.588190e-01,9.659258e-01,0.017213,0.999852
2,0.00,5.000000e-01,8.660254e-01,0.017213,0.999852
3,0.00,7.071068e-01,7.071068e-01,0.017213,0.999852
4,0.00,8.660254e-01,5.000000e-01,0.017213,0.999852
5,0.00,9.659258e-01,2.588190e-01,0.017213,0.999852
6,0.00,1.000000e+00,6.123234e-17,0.017213,0.999852
7,0.00,9.659258e-01,-2.588190e-01,0.017213,0.999852
8,0.00,8.660254e-01,-5.000000e-01,0.017213,0.999852
9,21.52,7.071068e-01,-7.071068e-01,0.017213,0.999852


In [105]:
# Define the search space for each model
def get_search_spaces():
    search_spaces = {
        'LinearRegression': {},  # No hyperparameters to tune

        'DecisionTreeRegressor': {
            'max_depth': Integer(3, 30, 'uniform'),
            'min_samples_split': Integer(2, 50, 'uniform'),
            'min_samples_leaf': Integer(1, 25, 'uniform'),
            'criterion': Categorical(['squared_error', 'friedman_mse', 'absolute_error']),
        },

        'RandomForestRegressor': {
            'n_estimators': Integer(100, 1000, 'uniform'),
            'max_depth': Integer(3, 30, 'uniform'),
            'min_samples_split': Integer(2, 50, 'uniform'),
            'min_samples_leaf': Integer(1, 25, 'uniform'),
            'bootstrap': Categorical([True, False]),
        },

        'LinearSVR': {
            'model__C': Real(0.01, 1000, 'log-uniform'),
            'model__epsilon': Real(0.001, 5, 'log-uniform'),
            'model__loss': Categorical(['epsilon_insensitive', 'squared_epsilon_insensitive']),
            'model__max_iter': Integer(10000, 100000, 'uniform'),
        },

        'SGDRegressor': {
            'model__alpha': Real(1e-6, 1, 'log-uniform'),
            'model__max_iter': Integer(100, 2000, 'uniform'),
            'model__tol': Real(1e-5, 1e-2, 'log-uniform'),
            'model__penalty': Categorical(['l2', 'l1', 'elasticnet']),
            'model__learning_rate': Categorical(['invscaling', 'constant', 'adaptive']),
        },

        'Ridge': {
            'model__alpha': Real(1e-4, 100, 'log-uniform'),
            'model__fit_intercept': Categorical([True, False]),
            'model__solver': Categorical(['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga']),
        },

        'Lasso': {
            'model__alpha': Real(1e-4, 100, 'log-uniform'),
            'model__fit_intercept': Categorical([True, False]),
            'model__selection': Categorical(['cyclic', 'random']),
        },

        'ElasticNet': {
            'model__alpha': Real(1e-4, 100, 'log-uniform'),
            'model__l1_ratio': Real(0.01, 1.0, 'uniform'),
            'model__fit_intercept': Categorical([True, False]),
            'model__selection': Categorical(['cyclic', 'random']),
        },

        'ExtraTreesRegressor': {
            'n_estimators': Integer(100, 1000, 'uniform'),
            'max_depth': Integer(3, 30, 'uniform'),
            'min_samples_split': Integer(2, 50, 'uniform'),
            'min_samples_leaf': Integer(1, 25, 'uniform'),
            'bootstrap': Categorical([True, False]),
        },

        'AdaBoostRegressor': {
            'n_estimators': Integer(100, 1000, 'uniform'),
            'learning_rate': Real(0.001, 0.5, 'log-uniform'),
            'loss': Categorical(['linear', 'square', 'exponential']),
        },

        'GradientBoostingRegressor': {
            'n_estimators': Integer(100, 1000, 'uniform'),
            'learning_rate': Real(0.001, 0.5, 'log-uniform'),
            'max_depth': Integer(3, 15, 'uniform'),
            'min_samples_split': Integer(2, 50, 'uniform'),
            'min_samples_leaf': Integer(1, 25, 'uniform'),
            'subsample': Real(0.5, 1.0, 'uniform'),
        },
    }
    return search_spaces

In [106]:
# Define models and their search spaces
# Models that require scaling
scaled_models = {
    'LinearSVR': (Pipeline([('scaler', StandardScaler()), ('model', LinearSVR())]), get_search_spaces()['LinearSVR']),
    'SGDRegressor': (Pipeline([('scaler', StandardScaler()), ('model', SGDRegressor())]), get_search_spaces()['SGDRegressor']),
    'Ridge': (Pipeline([('scaler', StandardScaler()), ('model', Ridge())]), get_search_spaces()['Ridge']),
    'Lasso': (Pipeline([('scaler', StandardScaler()), ('model', Lasso())]), get_search_spaces()['Lasso']),
    'ElasticNet': (Pipeline([('scaler', StandardScaler()), ('model', ElasticNet())]), get_search_spaces()['ElasticNet']),
}

# Models that do not require scaling
unscaled_models = {
    'LinearRegression': (LinearRegression(), get_search_spaces()['LinearRegression']),
    'DecisionTreeRegressor': (DecisionTreeRegressor(), get_search_spaces()['DecisionTreeRegressor']),
    'RandomForestRegressor': (RandomForestRegressor(), get_search_spaces()['RandomForestRegressor']),
    'ExtraTreesRegressor': (ExtraTreesRegressor(), get_search_spaces()['ExtraTreesRegressor']),
    'AdaBoostRegressor': (AdaBoostRegressor(), get_search_spaces()['AdaBoostRegressor']),
    'GradientBoostingRegressor': (GradientBoostingRegressor(), get_search_spaces()['GradientBoostingRegressor']),
}

# Combine all models
models = {**scaled_models, **unscaled_models}

In [107]:
# Perform BayesSearchCV for each model
results = {}
for name, (model, search_space) in models.items():
    if not search_space:
        # Skip models with no hyperparameters to tune
        results[name] = {'model': model, 'best_params': {}, 'best_score': None}
        continue

    opt = BayesSearchCV(
        estimator=model,
        search_spaces=search_space,
        n_iter=50,  # Number of iterations
        cv=3,  # 3-fold cross-validation
        scoring="neg_mean_absolute_error",
        n_jobs=-1,  # Use all CPU cores
        random_state=42,
    )

    # Fit the model (uncomment and replace with your data)
    opt.fit(X_train, y_train.values.ravel())

    results[name] = {
        'model': opt,
        'best_params': opt.best_params_,
        'best_score': opt.best_score_,
    }

/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iteration

In [108]:
# Print results
for name, result in results.items():
    print(f"{name}:")
    print(f"  Best parameters: {result['best_params']}")
    print(f"  Best score: {result['best_score']}")
    print()

LinearSVR:
  Best parameters: OrderedDict({'model__C': 1000.0, 'model__epsilon': 4.999999999999999, 'model__loss': 'epsilon_insensitive', 'model__max_iter': 10000})
  Best score: -7.920675690612068

SGDRegressor:
  Best parameters: OrderedDict({'model__alpha': 0.3179868274707896, 'model__learning_rate': 'constant', 'model__max_iter': 345, 'model__penalty': 'l1', 'model__tol': 0.00014726969511424812})
  Best score: -7.288299017309661

Ridge:
  Best parameters: OrderedDict({'model__alpha': 7.488174047459994, 'model__fit_intercept': True, 'model__solver': 'sparse_cg'})
  Best score: -8.467455040080608

Lasso:
  Best parameters: OrderedDict({'model__alpha': 4.519325173315875, 'model__fit_intercept': True, 'model__selection': 'random'})
  Best score: -7.983454638539224

ElasticNet:
  Best parameters: OrderedDict({'model__alpha': 0.03842760508187927, 'model__fit_intercept': True, 'model__l1_ratio': 1.0, 'model__selection': 'cyclic'})
  Best score: -8.45859044817716

LinearRegression:
  Best 

In [109]:
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
import numpy as np

# --- Step 1: Extract the best models from BayesSearchCV results ---
best_models = {}
for name, result in results.items():
    if result['best_params']:  # Skip models with no hyperparameters (e.g., LinearRegression)
        # Clone the best estimator and set its parameters
        best_model = clone(result['model'].best_estimator_)
        best_models[name] = best_model
    else:
        # For models without hyperparameters, use the original
        best_models[name] = result['model']

In [110]:
# --- Step 2: Evaluate Individual Models ---
individual_scores = {}
for name, model in best_models.items():
    scores = cross_val_score(model, X_train, y_train.values.ravel(), cv=3, scoring="neg_mean_absolute_error")
    individual_scores[name] = np.mean(scores)

print("\nIndividual Model Mean CV Scores:")
for name, score in individual_scores.items():
    print(f"{name}: {score}")

/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



Individual Model Mean CV Scores:
LinearSVR: -8.022993163700315
SGDRegressor: -8.457060186695125
Ridge: -8.467455040080608
Lasso: -8.023441762187147
ElasticNet: -8.45859044817716
LinearRegression: -8.48093903042041
DecisionTreeRegressor: -5.145166666666667
RandomForestRegressor: -4.656777198889579
ExtraTreesRegressor: -4.273739918845743
AdaBoostRegressor: -11.212341392426582
GradientBoostingRegressor: -3.8924311436983756


In [120]:
# Normalize scores to weights (invert and normalize)
inv_scores = {name: 1 / (1 + score) for name, score in individual_scores.items()}  # Avoid division by zero
total = sum(inv_scores.values())
weights = [inv_scores[name] / total for name in best_models]

In [121]:
# --- Step 3: Define Ensemble Models ---

# Voting Regressor (averages predictions of all models)
voting_regressor = VotingRegressor(
    estimators=[(name, model) for name, model in best_models.items()],
    n_jobs=-1,
)

In [122]:
voting_regressor_weights = VotingRegressor(
    estimators=[(name, model) for name, model in best_models.items()],
    n_jobs=-1,
    weights=weights,
)

In [123]:
# Stacking Regressor (uses a meta-model to combine predictions)
# Use a subset of models as base learners to avoid overfitting
base_models = [
    ('LinearSVR', best_models['LinearSVR']),
    ('GradientBoostingRegressor', best_models['GradientBoostingRegressor']),
    ('ExtraTreesRegressor', best_models['ExtraTreesRegressor']),
    ('Lasso', best_models['Lasso']),
]

In [124]:
# Meta-model (e.g., Linear Regression)
stacking_regressor = StackingRegressor(
    estimators=base_models,
    final_estimator=clone(best_models['RandomForestRegressor']),
    cv=3,  # Cross-validation for meta-model training
    n_jobs=-1
)

In [125]:
# Meta-model (e.g., Linear Regression)
stacking_regressor_all = StackingRegressor(
    estimators=[(name, model) for name, model in best_models.items()],
    final_estimator=clone(best_models['Ridge']),
    cv=3,  # Cross-validation for meta-model training
    n_jobs=-1
)

In [126]:
# --- Step 4: Train and Evaluate Ensembles ---

In [127]:
# Train Voting Regressor
voting_regressor.fit(X_train, y_train.values.ravel())


/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('LinearSVR', ...), ('SGDRegressor', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"epsilon epsilon: float, default=0.0Epsilon parameter in the epsilon-insensitive loss function. Notethat the value of this parameter depends on the scale of the targetvariable y. If unsure, set ``epsilon=0``.",4.999999999999999
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.",1000.0
,"loss loss: {'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='epsilon_insensitive'Specifies the loss function. The epsilon-insensitive loss(standard SVR) is the L1 loss, while the squared epsilon-insensitiveloss ('squared_epsilon_insensitive') is the L2 loss.",'epsilon_insensitive'


In [128]:
voting_regressor_weights.fit(X_train, y_train.values.ravel())

/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('LinearSVR', ...), ('SGDRegressor', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.","[np.float64(0....1348105149541), np.float64(0....3703073353485), ...]"
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"epsilon epsilon: float, default=0.0Epsilon parameter in the epsilon-insensitive loss function. Notethat the value of this parameter depends on the scale of the targetvariable y. If unsure, set ``epsilon=0``.",4.999999999999999
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.",1000.0
,"loss loss: {'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='epsilon_insensitive'Specifies the loss function. The epsilon-insensitive loss(standard SVR) is the L1 loss, while the squared epsilon-insensitiveloss ('squared_epsilon_insensitive') is the L2 loss.",'epsilon_insensitive'


In [129]:
# Train Stacking Regressor
stacking_regressor.fit(X_train, y_train.values.ravel())

/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('LinearSVR', ...), ('GradientBoostingRegressor', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",RandomForestR...mples_split=3)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",3
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"epsilon epsilon: float, default=0.0Epsilon parameter in the epsilon-insensitive loss function. Notethat the value of this parameter depends on the scale of the targetvariable y. If unsure, set ``epsilon=0``.",4.999999999999999
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001


In [131]:
# Train Stacking Regressor
stacking_regressor_all.fit(X_train, y_train.values.ravel())

/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('LinearSVR', ...), ('SGDRegressor', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",Pipeline(step...sparse_cg'))])
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",3
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"epsilon epsilon: float, default=0.0Epsilon parameter in the epsilon-insensitive loss function. Notethat the value of this parameter depends on the scale of the targetvariable y. If unsure, set ``epsilon=0``.",4.999999999999999
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001


In [132]:
# Evaluate using cross-validation
voting_scores = cross_val_score(voting_regressor, X_train, y_train.values.ravel(), cv=3, scoring="neg_mean_absolute_error")
voting_weights_scores = cross_val_score(voting_regressor_weights, X_train, y_train.values.ravel(), cv=3, scoring="neg_mean_absolute_error")
stacking_scores = cross_val_score(stacking_regressor, X_train, y_train.values.ravel(), cv=3, scoring="neg_mean_absolute_error")
stacking_all_scores = cross_val_score(stacking_regressor_all, X_train, y_train.values.ravel(), cv=3, scoring="neg_mean_absolute_error")


/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/dennis/Documents/machine-learning/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iteration

In [134]:
# Print results
print("Voting Regressor CV Scores:", voting_scores)
print("Mean CV Score:", np.mean(voting_scores))
print()
print("Voting Regressor weights CV Scores:", voting_weights_scores)
print("Mean CV Score:", np.mean(voting_weights_scores))
print()
print("Stacking Regressor CV Scores:", stacking_scores)
print("Mean CV Score:", np.mean(stacking_scores))
print()
print("Stacking Regressor all CV Scores:", stacking_all_scores)
print("Mean CV Score:", np.mean(stacking_all_scores))

Voting Regressor CV Scores: [-6.42037178 -9.55553411 -2.65891239]
Mean CV Score: -6.21160609319494

Voting Regressor weights CV Scores: [-5.47887928 -9.15625039 -2.01664823]
Mean CV Score: -5.550592635192441

Stacking Regressor CV Scores: [-4.85916299 -7.1928851  -1.3225619 ]
Mean CV Score: -4.458203330325849

Stacking Regressor all CV Scores: [-4.79480343 -8.86302841 -1.92647375]
Mean CV Score: -5.194768530797783


In [135]:
best_models

{'LinearSVR': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  LinearSVR(C=1000.0, epsilon=4.999999999999999,
                            max_iter=10000))]),
 'SGDRegressor': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  SGDRegressor(alpha=0.3179868274707896,
                               learning_rate='constant', max_iter=345,
                               penalty='l1', tol=0.00014726969511424812))]),
 'Ridge': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model', Ridge(alpha=7.488174047459994, solver='sparse_cg'))]),
 'Lasso': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model', Lasso(alpha=4.519325173315875, selection='random'))]),
 'ElasticNet': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model', ElasticNet(alpha=0.03842760508187927, l1_ratio=1.0))]),
 'LinearRegression': LinearRegression(),
 'DecisionTreeRegressor': DecisionTreeRegres

In [137]:
import joblib
joblib.dump(best_models, 'models/best_models.joblib')

['models/best_models.joblib']

In [ ]:
best_models = joblib.load('models/best_models.joblib')

gb_reg = best_models['GradientBoostingRegressor']

gb_reg.fit(X_train, y_train.values.ravel())

joblib.dump(gb_reg, 'models/fitted_gb_reg.joblib')